### MDF Forcing for local ARISE-AIM Experiments

This notebook generates netcdf files that house atmospheric and ocean variables read by Icepack as prescribed forcing for localized sea ice modeling experiments at **Cambridge Bay, NU** and **Vallunden Lagoon, Svalbard**. 

The **output files** are standardized netcdf files with dimensions ('time', 'loc') and a POSIX calendar basis in *minutes since 1970-01-01*. Since the MDF forcing module in Icepack will do data aggregation and interpolation internally, there is no need for this notebook to standardize the data input. Output data variables are:

*Atmosphere*
- 'tas', surface air temperature
- 'uas', zonal wind velocity
- 'vas', meridional wind velocity 
- 'hus', specific humidity
- 'rsds', downwelling shortwave radiative flux
- 'rlds', downwelling longwave radiative flux
- 'pr', precipitation

*Ocean*
- 'tos', sea surface temperature
- 'uo', zonal ocean velocity
- 'vo', meriodional ocean velocity
- 'so', sea water salinity
- 'mlotst', mixed layer depth
- 'hfsot', deep ocean heat flux



The **input files** are:
- local hourly atmospheric conditions at nearby weather stations (.csv, TZ: local and UTC timestamps provided)
- gridded (1deg) hourly CERES radiation data from the nearest proximal grid cell (.nc, TZ: GMT)
- monthly gridded ocean data from the nearest proximal ocean grid cell (.nc, TZ: UTC) 

with variables:

*Atmosphere - Station Data*
- 'TEMP', surface air temperature
- 'WIND_SPEED', mean wind speed (per h)
- 'WIND_DIRECTION', mean wind direction (per h)
- 'STATION_PRESSURE', station air pressure (to specific humidity)
- 'DEW_POINT_TEMPERATURE', station dew point temprature (to specific humidity)
- 'RELATIVE_HUMIDITY', station relative humidity (to specific humidity)
- 'PRECIP_AMOUNT', precipitation amount (in mm / h)

*Atmosphere - CERES Radiative Data*
- 'adj_atmos_sw_down_all_surface_1h', downwelling shortwave radiative flux
- 'adj_atmos_lw_down_all_surface_1h', downwelling longwave radiative flux

*Ocean*
- 'sosstsst', sea surface temperature
- 'vozocrtx', zonal ocean velocity
- 'vomecrty', meridional ocean velocity
- 'sosaline', sea water salinity
- 'somxl030', mixed layer depth
- 'sohefldo', deep ocean heat flux


In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta, datetime
from zoneinfo import ZoneInfo
from metpy.calc import specific_humidity_from_dewpoint
from metpy.calc import dewpoint_from_relative_humidity  
from metpy.units import units
from mdftoolkit.MDF_toolkit import MDF_toolkit

In [ ]:
input_data_path = '/Users/mollyw/Desktop/ARISE-AIM/Data/postprocessed/'
output_data_path = '/Users/mollyw/Desktop/ARISE-AIM/Data/mdf_forcings/'
station = 'Svalbard'

if station == 'CambridgeBay':
    station_file = input_data_path + '/atm/CambridgeBay_ATMOSPHERE.csv'
    ocean_file = input_data_path + '/ocn/CambridgeBay_OCEAN.nc'
    ceres_file = input_data_path + '/atm/CambridgeBay_CERES.nc'
elif station == 'Svalbard':
    station_file = input_data_path + '/atm/Svalbard_ATMOSPHERE.csv'
    ocean_file = input_data_path + '/ocn/Svalbard_OCEAN.nc'
    ceres_file = input_data_path + '/atm/Svalbard_CERES.nc'
else: 
    print('Reminder! Only experimental locations are Cambridge Bay or Svalbard!')

### Atmosphere - Station and CERES Data

In [66]:
def standardize_time(data):
    ''' 
    This function standardizes the time column of the data to a calendar basis that is expressed
    in minutes since 1970-01-01 00:00:00 UTC.

    To do so, the function 
    - identifies the time column in the data
    - converts time to a datetime object
    - adjusts the time to UTC if necessary
    - converts to a timestamp in minutes since 1970-01-01 00:00:00 UTC

    The function returns the data with the standardized time column (TIME).
    '''

    # Identify the time column in the data by searching on common element 'TIME'
    time_name = [key for key in data.keys() if ('TIME' in key)]
    if len(time_name) == 0:
        raise ValueError("No time column found in the data.")
    elif len(time_name) > 1:
        raise ValueError("Multiple time columns found in the data. Please specify which one to use.")
    else:
        time_name = time_name[0]
    
    # pull out the time column and convert it to a series of datetime objects
    if ('.' in data[time_name][0]):
        print('time conversion requires explicit formatting!')
        time = pd.to_datetime(data[time_name].values, format='%d.%m.%Y %H:%M')
    else:
        time = pd.to_datetime(data[time_name].values)

    assert len(time) == len(np.unique(time)), "there are duplicate time values (prior to UTC conversion)! please address this."
    
    
    # convert to UTC if necessary
    if 'UTC' in time_name:
        print('timezone already tz aware!')
        # time = time.tz_localize('UTC')
    else:
        timezone = ZoneInfo(time_name[6:-1])
        print('Timezone needs to be converted to UTC. Prevous timezone was ', str(timezone))
        print(time)
        time = time.tz_localize(timezone, ambiguous=True, nonexistent=pd.Timedelta(hours=1))
        time = time.tz_convert('UTC')
        
        idx = np.where(time.floor("s") == '2024-03-31 01:00:00')[0]
        if len(idx) > 1:
            idx = idx[1]
            time_sample = time[idx:]
            time_sample = time_sample + pd.Timedelta(hours=1)
            time = pd.to_datetime(list(time[:idx]) + list(time_sample))

        idx = np.where(time.floor("s") == '2025-03-30 02:00:00')[0]
        if len(idx) > 1:
            idx = idx[1]
            time_sample = time[idx:]
            time_sample = time_sample + pd.Timedelta(hours=1)
            time = pd.to_datetime(list(time[:idx]) + list(time_sample))

    assert time[0].tzinfo is not None, "Time conversion to UTC failed."

    # assert len(time) == len(np.unique(time)), "there are duplicate time values (after UTC conversion)! please address this."

    # save UTC time 
    data['TIME (UTC)'] = time

    # convert to a Timestamp in minutes since 1970-01-01 00:00:00 UTC
    bot = pd.to_datetime('1970-01-01 00:00:00').tz_localize('UTC')
    delta_ints = np.floor((time - bot).total_seconds()) / 60.0 # minutes   
    time_inds  = pd.Index(delta_ints, dtype='int64')

    # add the standardized time column to the data
    data['TIME (minutes since 1970)'] = time_inds.values

    return data


In [68]:
def get_specific_humidity(data):
    '''
    Function description: TBD
    '''

    if 'DEW_POINT_TEMP (degC)' in data.keys():
        # if the dewpoint temperature is already provided, use it
        dewpoint = data['DEW_POINT_TEMP (degC)'].values * units.degC
    else:
        # calculate the dewpoint temperature
        rh_key = [key for key in data.keys() if 'RELATIVE_HUMIDITY' in key]
        if len(rh_key) == 0:
            raise ValueError("No dewpoint temperature (degC) or relative humidity column found in the data.")
        elif len(rh_key) > 1:
            raise ValueError("Multiple relative humidity columns found in the data. Please specify which one to use.")
        else:
            rh_key = rh_key[0]
            # extract the units by splitting the key on white space
            rh_units = rh_key.split()[-1]
            relative_humidity = data[rh_key].values
        # check if the relative humidity is in percentage and convert to fraction if necessary
        if rh_units in ['%', '(%)']:
            # convert relative humidity from percentage to fraction
            relative_humidity = relative_humidity / 100
    
        dewpoint = dewpoint_from_relative_humidity(data['TEMP (degC)'].values * units.degC,
                                                   relative_humidity)
    
    # make sure pressure is in the correct unites of hPa
    pressure_key = [key for key in data.keys() if 'STATION_PRESSURE' in key]
    if len(pressure_key) == 0:
        raise ValueError("No station pressure column found in the data.")
    elif len(pressure_key) > 1:
        raise ValueError("Multiple station pressure columns found in the data. Please specify which one to use.")
    else:
        pressure_key = pressure_key[0]
        # extract the units by splitting the key on white space
        pressure_units = pressure_key.split()[-1]
    
    if pressure_units in ['kPa','(kPa)']:
        # convert pressure from kPa to hPa
         pressure = data[pressure_key].values * 10
    elif pressure_units not in ['hPa', '(hPa)']:
        raise ValueError(f"Unsupported pressure units: {pressure_units}. Please use 'hPa' or 'kPa'.")
    else:
        pressure = data[pressure_key].values
    
    # calculate the specific humidity from the dewpoint temperature and pressure
    dummy = specific_humidity_from_dewpoint(pressure * units.hPa, dewpoint).to('kg/kg')
    shum = np.reshape(np.array(dummy.magnitude), (len(pressure), 1))

    # add the specific humidity to the data
    data['SPECIFIC_HUMIDITY (kg/kg)'] = shum
   
    return data


In [69]:
def get_uv_wind(data):
    '''
    Function description: TBD

    Assumes that wind direction is given in degrees from the geographic North Pole, so 
    trig functions will be applied "backwards" due to degree orientation to N and E
   
    '''

    # Step 1. Determine units of wind speed and convert to m/s if necessary
    wind_speed_key = [key for key in data.keys() if 'WIND_SPEED' in key]
    if len(wind_speed_key) == 0:
        raise ValueError("No wind speed column found in the data.")
    elif len(wind_speed_key) > 1:
        raise ValueError("Multiple wind speed columns found in the data. Please specify which one to use.")
    else:
        wind_speed_key = wind_speed_key[0]
        # extract the units by splitting the key on white space
        wind_speed_units = wind_speed_key.split()[-1]
        wind_speed = data[wind_speed_key].values

    if wind_speed_units in ['km/h', '(km/h)']:
        # convert wind speed from km/h to m/s
        factor = 1000 / 3600
    elif wind_speed_units not in ['m/s', '(m/s)']:
        raise ValueError(f"Unsupported wind speed units: {wind_speed_units}. Please use 'm/s' or 'km/h'.")
    else:
        factor = 1  # no conversion needed, already in m/s

    wind_speed = wind_speed * factor

    # Step 2. Determine units of wind direction and convert to degrees in necessary
    wind_direction_key = [key for key in data.keys() if 'WIND_DIRECTION' in key]
    if len(wind_direction_key) == 0:
        raise ValueError("No wind direction column found in the data.")
    elif len(wind_direction_key) > 1:
        raise ValueError("Multiple wind direction columns found in the data. Please specify which one to use.")
    else:
        wind_direction_key = wind_direction_key[0]
        # extract the units by splitting the key on white space
        wind_direction_units = wind_direction_key.split()[-1]
        wind_direction = data[wind_direction_key].values
    
    if wind_direction_units in ['10deg', '(10deg)']:
        print('Wind direction is given in 10s of degrees relative to geo North. Converting to degrees.')
        # convert wind direction from 10's of degrees to degrees
        wind_direction = wind_direction * 10
    elif wind_direction_units not in ['deg', '(deg)']:
        raise ValueError(f"Unsupported wind direction units: {wind_direction_units}. Please use 'deg' or '10 deg'.")
    else:
        print('Wind direction is given in degrees relative to geo North. No conversion necessary.')

    # Step 3. If wind direction is 0, wind speed should be zero. Else, calculate U,V

    wind_u = np.zeros(len(wind_speed)) + np.nan
    wind_v = np.zeros(len(wind_speed)) + np.nan
    
    for k in range(len(wind_speed)):
        if (wind_direction[k] == 0) & (wind_speed[k] != 0):
            print('Wind speed recorded as ', wind_speed[k], ' at index k = ', k)
            print('when wind direction == 0.  Check data!')
        else:
            u = wind_speed[k] * np.sin(wind_direction[k])
            v = wind_speed[k] * np.cos(wind_direction[k])

        wind_u[k] = u 
        wind_v[k] = v 

    # Step 3. assign U,V to the data
    data['WIND_U (m/s)'] = wind_u
    data['WIND_V (m/s)'] = wind_v
    
    return data

In [70]:
def get_prec(data):
    '''
    Function description: TBD
    '''
    prec_key = [key for key in data.keys() if 'RAIN_AMOUNT' in key]
    snow_key = [key for key in data.keys() if 'SNOW_AMOUNT' in key]
    if len(prec_key) == 0:
        raise ValueError("No precipitation amount column found in the data.")
    elif len(prec_key) > 1:
        raise ValueError("Multiple precipitation amount columns found in the data. Please specify which one to use.")
    else:
        prec_key = prec_key[0]
        # extract the units by splitting the key on white space
        prec_units = prec_key.split()[-1]
        if prec_units in ['mm/hr', '(mm/hr)']:
            # convert precipitation amount from mm/hr to kg/m2/s
            factor = 1 / 3600
        elif prec_units not in ['mm/s', '(mm/s)', 'kg/m2/s', '(kg/m2/s)']:
            raise ValueError(f"Unsupported precipitation units: {prec_units}. Please use 'mm/s', 'kg/m2/s' or 'mm/hr'.")
        else:
            factor = 1

        if len(snow_key) > 0:
            snow_key = snow_key[0]
            snow_units = snow_key.split()[-1]
            if prec_units in ['mm/hr', '(mm/hr)']:
                # convert precipitation amount from mm/hr to kg/m2/s
                sfactor = 1 / 3600
            elif prec_units not in ['mm/s', '(mm/s)', 'kg/m2/s', '(kg/m2/s)']:
                raise ValueError(f"Unsupported precipitation units: {prec_units}. Please use 'mm/s', 'kg/m2/s' or 'mm/hr'.")
            else:
                sfactor = 1

        # convert precip and add it back into the column
        data['RAIN_AMOUNT (kg/m2/s)'] = data[prec_key].values * factor
        data['SNOW_AMOUNT (kg/m2/s)'] = data[snow_key].values * factor

    return data

In [71]:
def read_station_atmo(file, station, delimiter=','):

    # open the file and read the data
    df = pd.read_csv(file, index_col=False, delimiter=delimiter)

    # standardize the time column
    df = standardize_time(df)

    # reshape unmodified data for the xarray dataset
    # time = df['TIME (minutes since 1970)'].values
    time = df['TIME (UTC)'].values
    temp = df['TEMP (degC)'].values + 273.15

    # calculate derived variables from available observations
    df = get_uv_wind(df)
    df = get_specific_humidity(df)
    df = get_prec(df)

    wind_u = df['WIND_U (m/s)'].values
    wind_v = df['WIND_V (m/s)'].values
    shum = df['SPECIFIC_HUMIDITY (kg/kg)'].values
    prec = df['RAIN_AMOUNT (kg/m2/s)'].values
    prsn = df['SNOW_AMOUNT (kg/m2/s)'].values

    lat = df.y.values[0]
    lon = df.x.values[0]

    if station == 'CambridgeBay':
        desc_tag = 'Canadian govt weather station (GSN) in Cambridge Bay, NU, CA.'
        time_range = '2024-01-01 07:00:00 UTC through 2025-11-25 00:00:00 UTC'
    elif station == 'Svalbard':
        desc_tag = 'Norwegian govt weather station (SN99840) at Svalbard Lufthaven, NO.'
        time_range = '2024-01-01 01:00:00 CET through 2026-01-01 00:00:00 CET'
    else:
        raise ValueError("Unknown station name. Please use 'CambridgeBay' or 'Svalbard', or generate your own support.")
    
    
    time_atts = {'calendar'  : 'standard',}
    
    ds = xr.Dataset(data_vars = dict(tas=(['time'], temp, {'long_name': 'surface air temperature', 'units':'degC'}),
                                     uas = (['time'], wind_u, {'long_name': 'zonal wind', 'units':'m/s'}),
                                     vas = (['time'], wind_v, {'long_name': 'meridional wind', 'units':'m/s'}),
                                     hus = (['time'], shum, {'long_name': 'specific humidity', 'units':'g/kg'}),
                                     pr = (['time'], prec, {'long_name': 'precipitation (rain)', 'units':'kg/m2/s'}),
                                     prsn = (['time'], prsn, {'long_name': 'precipitation (snow)', 'units':'kg/m2/s'})),
                    coords = dict(lat = ([], lat),
                                  lon = ([], lon),
                                  time = (['time'], time, time_atts)),
                    attrs = dict(description='Hourly atmospheric data from the '+desc_tag,
                                 calendar= 'Time range is ' + time_range
                                 )
                    )   
    
    # ds = ds.resample(time='3h').mean()
    ds = ds.sel(time=slice('2024-01-01', '2025-12-31'))

    return ds

In [72]:
def read_ceres_fluxes(file, ds):
    '''
    Please note that the CERES data are only available up until 2025-03-01, annoyingly. The current work around for this
    is to assume little year-to-year variation and concatenate 2024-04-01:2024-07-01 to the end of the data set.'
    '''

    # determine time slice required by the data in the station dataset
    # station_time = [pd.to_datetime('1970-01-01 00:00:00') + pd.Timedelta(seconds = d*60) for d in ds.time.values]
    station_time = ds.time

    # open CERES file and extract data at the the corresponding times
    ds_ceres = xr.open_dataset(file).sel(time=station_time)
    ds_ceres = ds_ceres.sel(time=slice('2024-01-01', '2025-12-31'))

    # extract relevant variables 
    rlds = ds_ceres.adj_atmos_lw_down_all_surface_1h.values
    rsds = ds_ceres.adj_atmos_sw_down_all_surface_1h.values

    # add the relevant variables to the existing dataset
    ds['rsds'] = (['time'], rsds, {'long_name': 'CERES Adjusted All-Sky Profile Fluxes Shortwave Flux Down, Surface All-Sky conditions, Hourly Daily Means', 
                                          'units': 'W/m2', 'ceres_lat': str(ds_ceres.lat.values), 'ceres_lon': str(ds_ceres.lon.values)})
    ds['rlds'] = (['time'], rlds, {'long_name': 'CERES Adjusted All-Sky Profile Fluxes Longwave Flux Down, Surface All-Sky conditions, Hourly Daily Means', 
                                          'units': 'W/m2', 'ceres_lat': str(ds_ceres.lat.values), 'ceres_lon': str(ds_ceres.lon.values)})

    # ds['time'] = station_time

    return ds

In [73]:
ds = read_station_atmo(station_file, station)
ds = read_ceres_fluxes(ceres_file, ds)
df = ds.to_dataframe()
MDF = MDF_toolkit(supersite_name=station, multi_site=False)
MDF._beginning_of_time = pd.to_datetime('1970-01-01 00:00:00')
MDF.add_data_timeseries(df, cadence='time60')
MDF.write_files(output_dir='.')

time conversion requires explicit formatting!
Timezone needs to be converted to UTC. Prevous timezone was  Atlantic/Jan_Mayen
DatetimeIndex(['2024-01-01 01:00:00', '2024-01-01 02:00:00',
               '2024-01-01 03:00:00', '2024-01-01 04:00:00',
               '2024-01-01 05:00:00', '2024-01-01 06:00:00',
               '2024-01-01 07:00:00', '2024-01-01 08:00:00',
               '2024-01-01 09:00:00', '2024-01-01 10:00:00',
               ...
               '2025-12-31 15:00:00', '2025-12-31 16:00:00',
               '2025-12-31 17:00:00', '2025-12-31 18:00:00',
               '2025-12-31 19:00:00', '2025-12-31 20:00:00',
               '2025-12-31 21:00:00', '2025-12-31 22:00:00',
               '2025-12-31 23:00:00', '2026-01-01 00:00:00'],
              dtype='datetime64[ns]', length=17544, freq=None)
Wind direction is given in degrees relative to geo North. No conversion necessary.
[]
  ... there was no time series data provided for time01 cadence
[]
  ... there was no time seri

### Ocean - ORA5 Monthly Reanalysis Data

In [38]:
def standardize_ocn_time(data):
    ''' 
    This function standardizes the time column of the data to a calendar basis that is expressed
    in minutes since 1970-01-01 00:00:00 UTC.

    The function returns the data with the standardized time variable (TIME).
    '''

    # Alter time to a tz-aware object as a timestamp
    time = pd.DatetimeIndex(data.time.values)
    time = time.tz_localize('UTC')

    # add the standardized time column to the data
    data['time'] = time
    return data

In [54]:
ds = xr.open_dataset(ocean_file)
ds = standardize_ocn_time(ds)
ds['sosstsst'] = ds.sosstsst + 273.15

In [ ]:
ds = xr.Dataset(data_vars = dict(hfsot = (['time'], list(ds.sohefldo.values[:,0])),
                                   mlotst = (['time'], list(ds.somxl030.values[:,0])),
                                   so = (['time'], list(ds.sosaline.values[:,0])),
                                   tos = (['time'], list(ds.sosstsst.values[:,0])),
                                   vo = (['time'], list(ds.vomecrty.values[:,0])),
                                   uo = (['time'], list(ds.vozocrtx.values[:,0]))),
                  coords = dict(time=ds.time,
                                lat = ([], ds.lat.values[0]),
                                lon = ([], ds.lon.values[0])))

# resample to daily data
ds1 = ds.resample(time='1D').bfill()
ds2 = xr.concat([ds1.isel(time=slice(None, None, -1)), ds1], dim='time').rolling({'time':30}).mean()
ds = ds2.isel(time=slice(len(ds1.time),len(ds2.time)))


# add attribute information
ds["hfsot"].attrs["units"] = "W/m2"
ds["hfsot"].attrs["long_name"] = "Deep ocean heat flux"

ds["mlotst"].attrs["units"] = "m"
ds["mlotst"].attrs["long_name"] = "Mixed layer depth"

ds["so"].attrs["units"] = "psu"
ds["so"].attrs["long_name"] = "Sea water salinity"

ds["tos"].attrs["units"] = "K"
ds["tos"].attrs["long_name"] = "sea surface temperature"

ds["vo"].attrs["units"] = "m/s"
ds["vo"].attrs["long_name"] = "meridional ocean velocity"

ds["uo"].attrs["units"] = "m/s"
ds["uo"].attrs["long_name"] = "zonal ocean velocity"


In [56]:
thing = ds.sel(time=slice('2025-01-16', '2026-01-15'))
thing['time'] = thing.time + pd.Timedelta(days=365)

ds_1 = xr.concat([ds, thing], dim='time')

In [58]:
df = ds_1.to_dataframe()
MDF = MDF_toolkit(supersite_name=station+'_ocn', multi_site=False)
MDF._beginning_of_time = pd.to_datetime('1970-01-01 00:00:00')
MDF.add_data_timeseries(df, cadence='time1440')
MDF.write_files(output_dir='.')

[]
  ... there was no time series data provided for time01 cadence
[]
  ... there was no time series data provided for time02 cadence
[]
  ... there was no time series data provided for time05 cadence
[]
  ... there was no time series data provided for time10 cadence
[]
  ... there was no time series data provided for time15 cadence
[]
  ... there was no time series data provided for time30 cadence
[]
  ... there was no time series data provided for time60 cadence
[]
  ... there was no time series data provided for time180 cadence
[                              hfsot     mlotst         so         tos  \
time                                                                    
2024-01-15 00:00:00+00:00 -3.944478  15.215301  28.238089  271.626464   
2024-01-16 00:00:00+00:00 -3.944478  15.215301  28.238089  271.626464   
2024-01-17 00:00:00+00:00 -3.944478  15.215301  28.238089  271.626464   
2024-01-18 00:00:00+00:00 -3.944478  15.215301  28.238089  271.626464   
2024-01-19 00:00:00+00:0